# The full KF beside the banks, and the excess error

Two follow-ups Leszek asked for after the bank figures of `bank-skf-fkf.ipynb` (commit `57bd568`):

1. Run the full Kalman filter in both scenarios, to see whether whitening is what speeds up convergence.
2. Show the excess error $s_t = \boldsymbol{x}_t^{\mathsf T}(\boldsymbol{\theta}_t - \boldsymbol{w}_{t-1})$ as
   $10\log_{10}\big(\mathrm{E}[s_t^2]/\mathrm{E}[(\boldsymbol{x}_t^{\mathsf T}\boldsymbol{\theta}_t)^2]\big)$.

**Choices made with Ramiro before writing code (2026-10-06)**

| doubt | choice |
|---|---|
| "the full Kalman" | one full-covariance KF, eqs. (31)-(32) as numbered in `vkf-kf.ipynb` (`KF_batch`), with the exact Laplacian $\chi$ and $b_\eta = \mathrm{E}\lvert\eta_t\rvert$, tuned to $-20$ dB by notebook 09's search like the single filters of the bank notebook. Not a bank of full KFs, not the Gaussian Kalman filter |
| "both scenarios" | AR($-0.9$) and white input, the two scenarios of the bank notebook, each on notebook 09's change test |
| filters | the four run in both scenarios there (exact sKF, exact fKF, sKF bank, fKF bank) and the full KF. The minorized and matched filters move along $\boldsymbol{x}_t$ like the exact ones and differ only in $\chi$, so they would add nothing on whitening |
| where | this new notebook; `bank-skf-fkf.ipynb` stays as committed |

**The excess error.** $\boldsymbol{\theta}_t$ is the true response: $\boldsymbol{h}_o$ before the change, $-\boldsymbol{h}_o$
from it. $\boldsymbol{w}_{t-1}$ is the estimate before the update at step $t$ (for a bank, its output
$\hat{\boldsymbol{w}}_{t-1}$). Then $e_t = s_t + \eta_t$: $s_t$ is the part of the prediction error that comes from the
filter being wrong, without the noise. The misalignment $\lVert\boldsymbol{\theta}_t - \boldsymbol{w}_{t-1}\rVert^2$
counts the error in every direction alike; $s_t$ counts it only along the input. If $\boldsymbol{w}_{t-1}$ is taken
independent of $\boldsymbol{x}_t$,

$$\mathrm{E}[s_t^2] = \mathrm{E}\big[(\boldsymbol{\theta}_t - \boldsymbol{w}_{t-1})^{\mathsf T}\boldsymbol{R}_{xx}(\boldsymbol{\theta}_t - \boldsymbol{w}_{t-1})\big],
\qquad \mathrm{E}[(\boldsymbol{x}_t^{\mathsf T}\boldsymbol{\theta}_t)^2] = \boldsymbol{h}_o^{\mathsf T}\boldsymbol{R}_{xx}\boldsymbol{h}_o.$$

- White input, $\boldsymbol{R}_{xx} = \boldsymbol{I}$ and $\lVert\boldsymbol{h}_o\rVert = 1$: the ratio is the misalignment, and
  the two curves should coincide.
- AR($-0.9$): almost all the input power is at high frequencies, so error at low frequencies barely shows in $s_t$.
  The gap between the two curves is how much of the misalignment the prediction errors do not see. That is the
  draft's explanation for why the banks crawl with coloured input (bank notebook, section 9), not tested there.

**Estimation.** $\mathrm{E}$ is the mean over the $R = 20$ realisations at each step. The mean of 20 squares still
moves by more than 1 dB from one step to the next, so for the curves and the step counts both means are smoothed by
a centred moving average of 101 steps (12.6 ms) before the ratio, on each side of the change separately (a window
across it would mix the floor before with the +6 dB after). Step counts on the excess error are good to about
50 steps. Floors and means over the record use the unsmoothed means.

**Run time** about 45 min, most of it the KF's $-20$ dB search (11 values, 20 realisations, $M \times M$ covariances).

## 1. Imports

In [ ]:
import time

import numpy as np
from matplotlib import pyplot as plt
from scipy.ndimage import uniform_filter1d
from scipy.signal import lfilter
from scipy.special import gammaln, log_ndtr, logsumexp
from scipy.stats import gennorm
import rir_generator as rir

%config InlineBackend.figure_format = 'svg'
NOTEBOOK_START = time.time()
print("imports ready")

## 2. The scenario

Section 2 of `bank-skf-fkf.ipynb` (commit `57bd568`), nothing edited: notebook 09's constants and signals, with the
AR coefficient as a parameter. New below it: `clean_output`, $\boldsymbol{x}_t^{\mathsf T}\boldsymbol{\theta}_t$ for
every realisation (the denominator, formed as `signals` forms $y_t$), and `signal_power`,
$\boldsymbol{h}_o^{\mathsf T}\boldsymbol{R}_{xx}\boldsymbol{h}_o$ (notebook 09's line, from `noise_of`).

In [ ]:
# === copied from bank-skf-fkf.ipynb section 2 (commit 57bd568), NOT edited ===
M = 128                     # filter length / length of the impulse response
N = 96000                   # steps per run at rest; the change run has 2N
R = 20                      # independent realisations
WARMUP = 500                # AR samples discarded so the input starts stationary
BETA = 0.2                  # shape of the generalized Gaussian measurement noise
SNR_DB = 5.0                # nominal signal-to-noise ratio
VAR_THETA_0 = 2.0           # initial prior variance on each weight (ours; Leszek's is 1/M)
FS = 8000                   # sampling rate [Hz]
TARGET_DB = -20.0           # floor the single filters are tuned to
DRIFT_DB = 0.5              # settling rule of notebook 09

# === copied from leszek-setup.ipynb section 4 (commit adb148b), NOT edited ===
ROOM, T60, C_SOUND, SRC, MIC = [5, 10, 6], 0.2, 340, [1, 2.5, 2], [1, 1.5, 1]
ho = rir.generate(c=C_SOUND, fs=FS, r=MIC, s=SRC, L=ROOM, reverberation_time=T60, nsample=M).flatten()
ho = ho/np.linalg.norm(ho)
lags = np.abs(np.subtract.outer(np.arange(M), np.arange(M)))


def noise_of(ar_a):
    # Noise scale and b_eta = E|eta| at 5 dB for this input: notebook 09's lines, AR as a parameter.
    P_signal = ho @ (ar_a**lags if ar_a != 0 else np.eye(M)) @ ho
    var_eta = P_signal/10**(SNR_DB/10)
    scale_gg = np.sqrt(var_eta/np.exp(gammaln(3/BETA) - gammaln(1/BETA)))
    return scale_gg, scale_gg*np.exp(gammaln(2/BETA) - gammaln(1/BETA))


def signals(seed, ar_a, n, change_at=None):
    # generate_signals / generate_signals_change of notebook 09 (commit 6e6dabc), with the AR
    # coefficient, the length and the instant of the flip as parameters. Same draws, same order.
    scale_gg, _ = noise_of(ar_a)
    rng = np.random.default_rng(seed)
    u = np.sqrt(1 - ar_a**2)*rng.standard_normal(n + WARMUP)
    x = lfilter([1.0], [1.0, -ar_a], u)[WARMUP:]
    y = np.convolve(ho, x)[:n]
    if change_at is not None:
        y[change_at:] = -y[change_at:]
    eta = gennorm.rvs(BETA, scale=scale_gg, size=n, random_state=rng)
    return x, y + eta
# === end of the copied block ===


def noise_variance(ar_a):
    # The true noise variance, for the Gaussian likelihood: the variance of the generalized Gaussian.
    scale_gg, _ = noise_of(ar_a)
    return scale_gg**2*np.exp(gammaln(3/BETA) - gammaln(1/BETA))


def stack(sigs):
    return np.array([s[0] for s in sigs]), np.array([s[1] for s in sigs])


for ar_a in (-0.9, 0.0):
    scale_gg, b_eta = noise_of(ar_a)
    print(f"AR {ar_a:>4}: var_eta = {noise_variance(ar_a):.4e}, noise scale = {scale_gg:.4e}, "
          f"b_eta = E|eta| = {b_eta:.4e}")
# === end of the copied block ===


def clean_output(X, change_at=None):
    # x_t' theta_t for every row of X, as signals() forms y: theta = h_o, and -h_o from change_at on
    Y = np.array([np.convolve(ho, x)[:X.shape[1]] for x in X])
    if change_at is not None:
        Y[:, change_at:] = -Y[:, change_at:]
    return Y


def signal_power(ar_a):
    # h_o' R_xx h_o: the line of noise_of
    return ho @ (ar_a**lags if ar_a != 0 else np.eye(M)) @ ho


# Copied from bank-skf-fkf.ipynb section 7 (commit 57bd568).
def scenario_signals(ar_a):
    return (stack([signals(seed, ar_a, N) for seed in range(R)]),
            stack([signals(seed, ar_a, 2*N, change_at=N) for seed in range(R)]))

## 3. The filters

| piece | source |
|---|---|
| `log_mills`, `chi_laplacian`, `chi_gaussian`, `_roll_in`, `sKF_batch`, `fKF_batch` | `bank-skf-fkf.ipynb` section 3 (commit `57bd568`) |
| `KF_batch` | `vkf-kf.ipynb` section 4 (commit `058a9f9`) |
| `transition_matrix`, `log_density_laplacian`, `bank_step`, `bank_batch`, the grids | `bank-skf-fkf.ipynb` section 4 |

Options added, nothing else changed; the defaults reproduce the originals.
- `excess` (all four loops): also return $s_t = \boldsymbol{x}_t^{\mathsf T}(\boldsymbol{\theta}_t - \boldsymbol{w}_{t-1})$
  at every step, formed from the same `dw` $= \boldsymbol{w}_{t-1} - \boldsymbol{\theta}_t$ as the misalignment.
- `flip_at` (`KF_batch` only, the others had it): misalignment and $s_t$ against $-\boldsymbol{h}$ from that step on.

In [ ]:
# === IGNACIO: log_mills / chi_laplacian - copied verbatim, NOT edited ===
# source: branch ignacio/joint-vs-marginal-vs-minorized, notebooks/07_skf_conjunto.ipynb, commit 9406077,
#         via bank-skf-fkf.ipynb (commit 57bd568)
def log_mills(z):
    """log R(z), with R(z) = Phi(-z)/phi(z) the Mills ratio, eq. (40). R grows like e^{z^2/2} for
    negative z and overflows, so it is only ever handled through its logarithm."""
    return log_ndtr(-z) + 0.5*z**2 + 0.5*np.log(2*np.pi)


def chi_laplacian(e, sigma, b_eta):
    """Correction chi_t(e_t) and its slope chi'_t(e_t) for Laplacian noise, eqs. (39), (41) and (43)."""
    u = e/sigma                                        # u_t = e_t / sigma_t
    k_t = sigma/b_eta                                  # k_t = sigma_t / b_eta
    tau = sigma**2/b_eta                               # tau_t = sigma_t^2 / b_eta, the largest correction
    log_R_minus = log_mills(k_t - u)                   # log R(k_t - u_t)
    log_R_plus = log_mills(k_t + u)                    # log R(k_t + u_t)
    Lambda = np.tanh((log_R_minus - log_R_plus)/2)     # (R- - R+)/(R- + R+), in (-1, 1)
    chi = tau*Lambda                                   # chi = tau Lambda
    chi_slope = (2*k_t*np.exp(-np.logaddexp(log_R_minus, log_R_plus))   # 2k / (R- + R+)
                 - k_t**2*(1 - Lambda**2))                               # - k^2 (1 - Lambda^2)
    return chi, chi_slope
# === end of the copied block ===


# Copied from bank-skf-fkf.ipynb (commit 57bd568), from vkf-kf.ipynb.
def chi_gaussian(e, sigma, v_eta):
    """Eq. (38), the Gaussian instance. Used only for the Kalman cross-check of section 4."""
    frac = sigma**2/(sigma**2 + v_eta)
    return frac*e, frac


# sKF_batch and fKF_batch: copied from bank-skf-fkf.ipynb (commit 57bd568), where they carry the options below;
# one option added here, `excess`, and nothing else changed.
#   flip_at: misalignment against -h from that step on (fkf.ipynb had it for the fKF only)
#   clip:    chi' -> max(chi', 0) in the variance update, eq. (36); the mean update is untouched
#   record:  also return chi'_t and v_t at every step
#   v0:      initial v (VAR_THETA_0 = 2 in every notebook of ours; 1/M in Leszek's ggbench.py)
#   wait:    first step that updates (M: wait for a full window, as every notebook of ours)
#   excess:  also return s_t = x_t'(theta_t - w_{t-1}) at every step (not with record)
def _roll_in(X_t, x_win):
    x_win = np.roll(x_win, 1, axis=1)
    x_win[:, 0] = X_t
    return x_win


def sKF_batch(X, D, h, b_eta, epsilon, chi_fn, flip_at=None, clip=False, record=False,
              v0=VAR_THETA_0, wait=None, excess=False):
    L, (B, n) = len(h), X.shape
    first = L if wait is None else wait
    w, x_t = np.zeros((B, L)), np.zeros((B, L))
    v = np.full(B, float(v0))
    eps = np.asarray(epsilon, dtype=float)
    mis = np.empty((B, n))
    if excess:
        s = np.empty((B, n))
    if record:
        slope_hist, v_hist = np.full((B, n), np.nan), np.full((B, n), np.nan)
    for t in range(n):
        x_t = _roll_in(X[:, t], x_t)
        e = D[:, t] - np.einsum("bm,bm->b", x_t, w)
        dw = (w - h) if flip_at is None or t < flip_at else (w + h)
        mis[:, t] = np.einsum("bm,bm->b", dw, dw)
        if excess:
            s[:, t] = -np.einsum("bm,bm->b", x_t, dw)          # s_t = x_t'(theta_t - w_{t-1})
        if t < first:
            continue
        v_tilde = v + eps
        power = np.einsum("bm,bm->b", x_t, x_t)
        sigma = np.sqrt(v_tilde*power)
        chi, slope = chi_fn(e, sigma, b_eta)
        w = w + x_t*(chi/power)[:, None]
        v = v_tilde*(1 - (np.maximum(slope, 0.0) if clip else slope)/L)
        if record:
            slope_hist[:, t], v_hist[:, t] = slope, v
    if excess:
        return mis, s
    if record:
        return mis, slope_hist, v_hist
    return mis


def fKF_batch(X, D, h, b_eta, v, chi_fn, flip_at=None, wait=None, excess=False):
    """B fKFs of eq. (37) side by side. Misalignment against h, or against -h from step flip_at on."""
    L, (B, n) = len(h), X.shape
    first = L if wait is None else wait
    w, x_t = np.zeros((B, L)), np.zeros((B, L))
    v = np.asarray(v, dtype=float)
    mis = np.empty((B, n))
    if excess:
        s = np.empty((B, n))
    for t in range(n):
        x_t = _roll_in(X[:, t], x_t)
        e = D[:, t] - np.einsum("bm,bm->b", x_t, w)
        dw = (w - h) if flip_at is None or t < flip_at else (w + h)
        mis[:, t] = np.einsum("bm,bm->b", dw, dw)
        if excess:
            s[:, t] = -np.einsum("bm,bm->b", x_t, dw)          # s_t = x_t'(theta_t - w_{t-1})
        if t < first:
            continue
        power = np.einsum("bm,bm->b", x_t, x_t)
        chi, _ = chi_fn(e, np.sqrt(v*power), b_eta)
        w = w + x_t*(chi/power)[:, None]
    if excess:
        return mis, s
    return mis


# KF_batch: copied from vkf-kf.ipynb (commit 058a9f9); options flip_at and excess added (as in sKF_batch above),
# nothing else changed.
def KF_batch(X, D, h, b_eta, epsilon, chi_fn=chi_laplacian, flip_at=None, excess=False):
    L, (B, n) = len(h), X.shape
    w, x_t = np.zeros((B, L)), np.zeros((B, L))
    Sigma = np.broadcast_to(np.eye(L)*float(VAR_THETA_0), (B, L, L)).copy()
    eps_I = np.asarray(epsilon, dtype=float)[:, None, None]*np.eye(L)
    mis, k_hist = np.empty((B, n)), np.zeros((B, n))
    if excess:
        s = np.empty((B, n))
    for t in range(n):
        x_t = _roll_in(X[:, t], x_t)
        e = D[:, t] - np.einsum("bm,bm->b", x_t, w)
        dw = (w - h) if flip_at is None or t < flip_at else (w + h)
        mis[:, t] = np.einsum("bm,bm->b", dw, dw)
        if excess:
            s[:, t] = -np.einsum("bm,bm->b", x_t, dw)          # s_t = x_t'(theta_t - w_{t-1})
        if t < L:
            continue
        St = Sigma + eps_I
        Sx = np.einsum("bij,bj->bi", St, x_t)
        sigma2 = np.einsum("bi,bi->b", x_t, Sx)
        sigma = np.sqrt(sigma2)
        k_hist[:, t] = sigma/b_eta
        chi, slope = chi_fn(e, sigma, b_eta)
        w = w + Sx*(chi/sigma2)[:, None]
        Sigma = St - (slope/sigma2)[:, None, None]*np.einsum("bi,bj->bij", Sx, Sx)
        Sigma = 0.5*(Sigma + np.swapaxes(Sigma, 1, 2))
    if excess:
        return mis, k_hist, s
    return mis, k_hist


# === copied from bank-skf-fkf.ipynb section 4 (commit 57bd568); bank_batch has the option excess added ===
LOG_2PI = np.log(2*np.pi)


def transition_matrix(K, q):
    """p_ij of eq. (eps.chain): q to each neighbour on the grid, 1 - q n_i to stay."""
    P = np.zeros((K, K))
    for i in range(K):
        neighbours = [j for j in (i - 1, i + 1) if 0 <= j < K]   # n_i = 1 at the ends, 2 inside
        P[i, neighbours] = q
        P[i, i] = 1 - q*len(neighbours)
    return P


def log_density_laplacian(e, sigma, b_eta):
    """log p_t(e) of eq. (bank.weights), from eq. (NL.density): log phi(u) + log(R(k-u) + R(k+u)) - log 2b."""
    u = e/sigma                                        # u = e / sigma
    k_t = sigma/b_eta                                  # k = sigma / b_eta
    return (-0.5*u**2 - 0.5*LOG_2PI                    # log phi(u)
            + np.logaddexp(log_mills(k_t - u), log_mills(k_t + u))   # log(R(k - u) + R(k + u))
            - np.log(2*b_eta))


def bank_step(W, V, log_mu, x_t, d_t, power, grid, log_P, chi_fn, log_density, scale):
    """One step of Algorithm 1 (V given) or Algorithm 2 (V = None) for B banks side by side.
    W (K, B, L) locations, V (K, B) variances, log_mu (K, B) log weights, x_t (B, L), d_t and power (B,).
    Returns the new W, V, log_mu, and log c, log p and u = e/sigma of this step (K, B)."""
    L = W.shape[-1]
    # mixing, eqs. (bank.mixing.prob) and (bank.mixing)
    log_joint = log_P[:, :, None] + log_mu[:, None, :]   # log p_ij mu_{t-1,i}, (i, j, B)
    log_c = logsumexp(log_joint, axis=0)                 # log c_{t,j}
    mix = np.exp(log_joint - log_c[None])                # mu_{t-1,i|j}
    W_bar = np.einsum("ijb,ibm->jbm", mix, W)            # w_bar_{t,j}
    if V is not None:
        diff = W[:, None] - W_bar[None]                  # w_{t-1,i} - w_bar_{t,j}, (i, j, B, L)
        spread = np.einsum("ijbm,ijbm->ijb", diff, diff)/L
        V_bar = np.einsum("ijb,ijb->jb", mix, V[:, None] + spread)   # v_bar_{t,j}
        v_tilde = V_bar + grid[:, None]                  # v_tilde = v_bar + eps^(j)
    else:
        v_tilde = np.broadcast_to(grid[:, None], log_mu.shape)       # fKF: the fixed v^(j)
    # one sKF / fKF step per filter, eq. (bank.update)
    sigma = np.sqrt(v_tilde*power)                       # sigma_{t,j}
    e = d_t - np.einsum("jbm,bm->jb", W_bar, x_t)        # e_{t,j}, each filter's own error
    chi, slope = chi_fn(e, sigma, scale)
    W = W_bar + x_t*(chi/power)[..., None]
    if V is not None:
        V = v_tilde*(1 - slope/L)
    # weights, eq. (bank.weights), in logs
    log_p = log_density(e, sigma, scale)
    ell = log_c + log_p
    log_mu = ell - logsumexp(ell, axis=0)
    return W, V, log_mu, log_c, log_p, e/sigma


def bank_batch(X, D, h, grid, q, family, chi_fn, log_density, scale, v0=VAR_THETA_0, flip_at=None,
               wait=None, record=False, clamp=None, excess=False):
    """B banks side by side, one per row of X and D. Returns the misalignment of the output (B, n),
    the weights averaged over the rows (K, n) and the largest |u| = |e|/sigma met. record=True adds,
    per filter and step, log mu, log c, log p and the misalignment of each filter (K, B, n), and the
    output after each step (B, n, L). excess=True adds s_t = x_t'(theta_t - w_hat_{t-1}) (B, n)."""
    L, (B, n) = len(h), X.shape
    first = L if wait is None else wait
    grid = np.asarray(grid, dtype=float)
    K = len(grid)
    with np.errstate(divide="ignore"):
        log_P = np.log(transition_matrix(K, q))          # log 0 = -inf where p_ij = 0
    W = np.zeros((K, B, L))
    V = np.full((K, B), float(v0)) if family == "sKF" else None
    log_mu = np.full((K, B), -np.log(K))                 # mu_0 = 1/K
    x_t, w_hat = np.zeros((B, L)), np.zeros((B, L))
    mis = np.empty((B, n))
    if excess:
        s = np.empty((B, n))
    mu_mean = np.empty((K, n))
    u_max = 0.0
    if record:
        rec = {key: np.full((K, B, n), np.nan) for key in ("log_mu", "log_c", "log_p", "mis_j")}
        rec["w_hat"] = np.full((B, n, L), np.nan)
    for t in range(n):
        x_t = _roll_in(X[:, t], x_t)
        target = h if flip_at is None or t < flip_at else -h
        dw = w_hat - target
        mis[:, t] = np.einsum("bm,bm->b", dw, dw)
        if excess:
            s[:, t] = -np.einsum("bm,bm->b", x_t, dw)          # s_t = x_t'(theta_t - w_hat_{t-1})
        if record:
            dW = W - target
            rec["mis_j"][:, :, t] = np.einsum("jbm,jbm->jb", dW, dW)
        if t >= first:
            power = np.einsum("bm,bm->b", x_t, x_t)
            W, V, log_mu, log_c, log_p, u = bank_step(W, V, log_mu, x_t, D[:, t], power, grid, log_P,
                                                      chi_fn, log_density, scale)
            if clamp is not None:                        # Leszek's mu = max(mu, clamp), renormalized
                log_mu = np.maximum(log_mu - log_mu.max(axis=0), np.log(clamp))
                log_mu = log_mu - logsumexp(log_mu, axis=0)
            w_hat = np.einsum("jb,jbm->bm", np.exp(log_mu), W)          # eq. (bank.output)
            u_max = max(u_max, float(np.abs(u).max()))
            if record:
                rec["log_c"][:, :, t], rec["log_p"][:, :, t], rec["w_hat"][:, t] = log_c, log_p, w_hat
        mu_mean[:, t] = np.exp(log_mu).mean(axis=1)
        if record:
            rec["log_mu"][:, :, t] = log_mu
    out = {"mis": mis, "mu": mu_mean, "u_max": u_max}
    if excess:
        out["s"] = s
    if record:
        out.update(rec)
    return out


EPS_BANK = np.geomspace(1e-9, 1e-4, 6)     # the draft's remark, line 1614
V_BANK = np.geomspace(1e-6, 1e-2, 9)       # Leszek's fkfbank*.py
Q_BANK = 1e-4
# === end of the copied block ===

print("filters ready")

## 4. Checks

### 4.1 The KF and $s_t$ against a textbook Kalman filter

With the Gaussian $\chi$ (eq. 38, $v_\eta$ = the true noise variance), `KF_batch` must be the textbook Kalman filter
for the random-walk model. `kalman_reference` is copied from `vkf-kf.ipynb` (commit `058a9f9`), where it was written
without reference to eqs. (31)-(32). It keeps $\boldsymbol{w}_{t-1}$ at every step, so the misalignment, $s_t$ and
$\boldsymbol{x}_t^{\mathsf T}\boldsymbol{\theta}_t$ are formed from it here, with windows cut directly from $x$ and not
through `_roll_in`. Realisations 0-2 of AR($-0.9$), 3000 steps, $\boldsymbol{h}_o \to -\boldsymbol{h}_o$ at step 1500,
$\varepsilon = 10^{-6}$.

In [ ]:
# === copied from vkf-kf.ipynb (commit 058a9f9), NOT edited ===
def shift(new_x_sample, x_window):
    L = len(x_window)
    new_x_window = np.zeros(L)
    new_x_window[0] = new_x_sample
    new_x_window[1:] = x_window[:-1]
    return new_x_window


def kalman_reference(n, x, d, w0, parameters):
    """Textbook Kalman filter for w_t = w_{t-1} + noise, written without reference to eqs. (31)-(32)."""
    epsilon, v_eta = parameters["epsilon"], parameters["v_eta"]
    L = len(w0)
    w, P = w0.copy(), np.eye(L)*float(parameters["var_theta_0"])
    x_t = np.zeros(L)
    w_hist = np.zeros((n, L))
    for t in range(n):
        x_t = shift(x[t], x_t)
        e = d[t] - x_t @ w
        w_hist[t] = w
        if t >= L:
            P_pred = P + epsilon*np.eye(L)
            S = x_t @ P_pred @ x_t + v_eta
            K = P_pred @ x_t/S
            w = w + K*e
            P = P_pred - np.outer(K, x_t @ P_pred)
    return {"w_hist": w_hist}
# === end of the copied block ===


def windows_of(X):
    # (B, n, M): row t of realisation b is (x_t, x_{t-1}, ..., x_{t-M+1}), zeros before the record
    X_pad = np.concatenate([np.zeros((len(X), M - 1)), X], axis=1)
    return np.lib.stride_tricks.sliding_window_view(X_pad, M, axis=1)[..., ::-1]


def theta_of(n, flip_at):
    # (n, M): theta_t = h_o, and -h_o from flip_at on
    return np.where(np.arange(n)[:, None] < flip_at, ho, -ho)


n_chk, flip_chk = 3000, 1500
X_chk, D_chk = stack([signals(seed, -0.9, n_chk, change_at=flip_chk) for seed in range(3)])
v_eta_chk = noise_variance(-0.9)
mis_kf, _, s_kf = KF_batch(X_chk, D_chk, ho, v_eta_chk, np.full(3, 1e-6), chi_gaussian, flip_at=flip_chk,
                           excess=True)
clean_chk = clean_output(X_chk, change_at=flip_chk)
win_chk, theta_chk = windows_of(X_chk), theta_of(n_chk, flip_chk)
gap_w = gap_mis = gap_s = gap_c = 0.0
for r in range(3):
    w_hist = kalman_reference(n_chk, X_chk[r], D_chk[r], np.zeros(M),
                              {"epsilon": 1e-6, "v_eta": v_eta_chk, "var_theta_0": VAR_THETA_0})["w_hist"]
    mis_ref = ((w_hist - theta_chk)**2).sum(axis=1)
    s_ref = np.einsum("tm,tm->t", win_chk[r], theta_chk - w_hist)
    clean_ref = np.einsum("tm,tm->t", win_chk[r], theta_chk)
    gap_mis = max(gap_mis, np.max(np.abs(mis_kf[r] - mis_ref)/mis_ref))
    gap_s = max(gap_s, np.max(np.abs(s_kf[r] - s_ref))/np.max(np.abs(s_ref)))
    gap_c = max(gap_c, np.max(np.abs(clean_chk[r] - clean_ref))/np.max(np.abs(clean_ref)))
print("KF_batch (Gaussian chi) against the textbook Kalman filter, 3 realisations, flip at step 1500:")
print(f"   misalignment, max relative difference      {gap_mis:.1e}")
print(f"   s_t, max |difference| / max |s_t|           {gap_s:.1e}")
print(f"   clean_output against x_t' theta_t from the windows: max |difference| / max |x_t' theta_t| {gap_c:.1e}")

### 4.2 The noise and the denominator

$d_t - \boldsymbol{x}_t^{\mathsf T}\boldsymbol{\theta}_t$ must be the noise $\eta_t$, drawn again as `signals` draws it
(as section 8.1 of the bank notebook does), and the mean of $(\boldsymbol{x}_t^{\mathsf T}\boldsymbol{\theta}_t)^2$ over a
run must be close to $\boldsymbol{h}_o^{\mathsf T}\boldsymbol{R}_{xx}\boldsymbol{h}_o$. Realisations 0-3 of the change run
of each scenario, $2N$ steps.

In [ ]:
def noise_draw(seed, ar_a, n):
    # eta as signals() draws it: the same generator, the input's draws first
    rng = np.random.default_rng(seed)
    rng.standard_normal(n + WARMUP)
    return gennorm.rvs(BETA, scale=noise_of(ar_a)[0], size=n, random_state=rng)


for ar_a in (-0.9, 0.0):
    X_n, D_n = stack([signals(seed, ar_a, 2*N, change_at=N) for seed in range(4)])
    clean_n = clean_output(X_n, change_at=N)
    eta_n = np.array([noise_draw(seed, ar_a, 2*N) for seed in range(4)])
    print(f"AR {ar_a:>4}: max |d - x'theta - eta| = {np.max(np.abs(D_n - clean_n - eta_n)):.1e} "
          f"(max |eta| = {np.max(np.abs(eta_n)):.1e}); mean (x'theta)^2 = {np.mean(clean_n[:, M:]**2):.4f}, "
          f"h' R_xx h = {signal_power(ar_a):.4f}")
del X_n, D_n, clean_n, eta_n

### 4.3 $s_t$ of the banks

A bank run with `record=True` keeps its output $\hat{\boldsymbol{w}}_t$ after every step, so
$s_t = \boldsymbol{x}_t^{\mathsf T}(\boldsymbol{\theta}_t - \hat{\boldsymbol{w}}_{t-1})$ can be formed from it and compared with
the one `bank_batch` returns. And a bank with $K = 1$ is the single filter bit for bit (check 1 of the bank notebook),
so its $s_t$ must equal that of `sKF_batch` / `fKF_batch`. Realisations 0-3 of AR($-0.9$), 6000 steps, flip at 3000.

In [ ]:
X_b, D_b = stack([signals(seed, -0.9, 6000, change_at=3000) for seed in range(4)])
lik_b = (chi_laplacian, log_density_laplacian, noise_of(-0.9)[1])
win_b, theta_b = windows_of(X_b), theta_of(6000, 3000)
for family, grid, value in (("sKF", EPS_BANK, 1.3493e-6), ("fKF", V_BANK, 2.985e-4)):
    out = bank_batch(X_b, D_b, ho, grid, Q_BANK, family, *lik_b, flip_at=3000, record=True, excess=True)
    w_prev = np.concatenate([np.zeros((4, 1, M)), np.nan_to_num(out["w_hat"][:, :-1], nan=0.0)], axis=1)
    s_ref = np.einsum("btm,btm->bt", win_b, theta_b - w_prev)          # w_hat = 0 before the first update
    single = (sKF_batch(X_b, D_b, ho, lik_b[2], np.full(4, value), chi_laplacian, flip_at=3000, excess=True)
              if family == "sKF" else
              fKF_batch(X_b, D_b, ho, lik_b[2], np.full(4, value), chi_laplacian, flip_at=3000, excess=True))
    one = bank_batch(X_b, D_b, ho, [value], Q_BANK, family, *lik_b, flip_at=3000, excess=True)
    print(f"{family} bank: s_t against the recorded outputs, max |difference| / max |s_t| = "
          f"{np.max(np.abs(out['s'] - s_ref))/np.max(np.abs(s_ref)):.1e}; K = 1 against the single filter: "
          f"s_t {'identical' if np.array_equal(one['s'], single[1]) else 'DIFFERENT'}, misalignment "
          f"{'identical' if np.array_equal(one['mis'], single[0]) else 'DIFFERENT'}")
del out, w_prev, s_ref, win_b

## 5. Measurement

Copied from `bank-skf-fkf.ipynb` section 7 (commit `57bd568`), nothing edited: notebook 09's `steady_state`,
`is_settled`, `recovery`, and `time_to`, `first_below`, `measure`, `show_table`. The KF's $-20$ dB value comes from
the search of the bank notebook's section 7.2 (`pick`, copied, and notebook 09's grid logspace$(-8, -3, 11)$, settled
points only, interpolated). `sweep_kf` runs it one grid value at a time: with $B = R$ the KF's $B \times M \times M$
covariances run about 25 % faster than all 220 at once.

New, for the excess error:
- `measure_excess`: floor at rest (last quarter, as the misalignment floor), first step below $-17$ and $-20$ dB at
  rest, first step after the change below $0, -5, -10, -15, -20$ dB, and the mean over the $2N$ record, all on
  $\mathrm{E}[s_t^2]/\mathrm{E}[(\boldsymbol{x}_t^{\mathsf T}\boldsymbol{\theta}_t)^2]$.
- `show_excess`: those numbers, with the misalignment's floor and mean beside them.

In [ ]:
# === IGNACIO: steady_state, is_settled and recovery - copied verbatim, NOT edited ===
# source: branch ignacio/joint-vs-marginal-vs-minorized, notebooks/09_matcheado.ipynb, commit 6e6dabc,
#         via bank-skf-fkf.ipynb (commit 57bd568)
def steady_state(misalignment):
    """Floor in dB, and the first step within 3 dB of it."""
    tail = slice(3*len(misalignment)//4, len(misalignment))
    floor = 10*np.log10(misalignment[tail].mean())
    db = 10*np.log10(misalignment)
    reached = int(np.argmax(db < floor + 3))
    return floor, reached


def is_settled(misalignment, drift):
    """Settled within the run: within 3 dB of the floor by half of it, and no longer descending."""
    n = len(misalignment)
    floor, reached = steady_state(misalignment)
    third_quarter = 10*np.log10(misalignment[n//2:3*n//4].mean())
    # reached = 0: the run never left the 3 dB band around its start, so it never converged either
    return bool(0 < reached <= n/2 and third_quarter - floor <= drift)


def recovery(misalignment, floor):
    """Steps from the change until the misalignment is back within 3 dB of the floor at rest;
    -1 if it never gets there before the run ends."""
    db = 10*np.log10(misalignment[N:])
    back = db < floor + 3
    if not back.any():
        return -1
    return int(np.argmax(back))
# === end of the copied block ===


# === copied from bank-skf-fkf.ipynb sections 7 and 7.2 (commit 57bd568), NOT edited ===
def time_to(misalignment, level_db):
    """First step after the change below level_db; -1 if never."""
    below = 10*np.log10(misalignment[N:]) < level_db
    return int(np.argmax(below)) if below.any() else -1


LEVELS = [0.0, -5.0, -10.0, -15.0]


def first_below(misalignment, level_db):
    below = 10*np.log10(misalignment) < level_db
    return int(np.argmax(below)) if below.any() else -1


def measure(rest_curve, change_curve):
    floor, reached = steady_state(rest_curve)
    return dict(floor=floor, reached=reached, settled=is_settled(rest_curve, DRIFT_DB),
                to_17=first_below(rest_curve, TARGET_DB + 3), to_20=first_below(rest_curve, TARGET_DB),
                levels=[time_to(change_curve, level) for level in LEVELS], rec=recovery(change_curve, floor),
                back_20=time_to(change_curve, TARGET_DB), mean_db=10*np.log10(change_curve.mean()),
                curve=change_curve)


def show_table(rows):
    print(f"{'':<34}{'at rest':-^46}  {'after the flip':-^50}")
    print(f"{'filter':<22}{'eps or v':>12}{'floor':>8}{'settled':>9}{'own+3':>7}{'-17 dB':>8}{'-20 dB':>8}"
          + "".join(f"{level:>8.0f}" for level in LEVELS) + f"{'own+3':>9}{'-20 dB':>9}{'mean [dB]':>11}")
    for name, r in rows.items():
        value = "grid" if np.isnan(r["value"]) else f"{r['value']:.4e}"
        print(f"{name:<22}{value:>12}{r['floor']:>8.2f}{'yes' if r['settled'] else 'NO':>9}{r['reached']:>7d}"
              f"{r['to_17']:>8d}{r['to_20']:>8d}" + "".join(f"{s:>8d}" for s in r["levels"])
              + f"{r['rec']:>9d}{r['back_20']:>9d}{r['mean_db']:>11.2f}")
    print("   steps; -1 = never within the run")


EPS_GRID = np.logspace(-8, -3, 11)          # notebook 09


def pick(grid, floors, settled, target):
    values, kept = grid[settled], floors[settled]
    if target < kept.min():
        return values[np.argmin(kept)], "target below every settled floor"
    if target > kept.max():
        return values[np.argmax(kept)], "target above every settled floor"
    order = np.argsort(kept)
    return 10**np.interp(target, kept[order], np.log10(values)[order]), "ok"
# === end of the copied block ===


def sweep_kf(b_eta, X, D):
    # notebook 09's search for the KF, as sweep() of the bank notebook, one grid value at a time
    floors, settled = [], []
    for eps in EPS_GRID:
        curve = KF_batch(X, D, ho, b_eta, np.full(len(X), eps))[0].mean(axis=0)
        floors.append(steady_state(curve)[0])
        settled.append(is_settled(curve, DRIFT_DB))
    return np.array(floors), np.array(settled)


SMOOTH = 101                # centred moving average on both means, in steps


def smoothed_ratio(num, den, cut=None):
    """Moving average of both means, then the ratio. No window crosses step cut (the change)."""
    if cut is not None:
        return np.concatenate([smoothed_ratio(num[:cut], den[:cut]), smoothed_ratio(num[cut:], den[cut:])])
    return uniform_filter1d(num, SMOOTH, mode="nearest")/uniform_filter1d(den, SMOOTH, mode="nearest")


def measure_excess(s_rest, s_change, den_rest, den_change):
    """s_rest (R, N), s_change (R, 2N); den_* = E[(x_t' theta_t)^2], the mean over realisations at each step."""
    num_rest, num_change = (s_rest**2).mean(axis=0), (s_change**2).mean(axis=0)
    rest_curve, change_curve = smoothed_ratio(num_rest, den_rest), smoothed_ratio(num_change, den_change, cut=N)
    tail = slice(3*N//4, N)                            # the window of steady_state
    return dict(floor=10*np.log10(num_rest[tail].mean()/den_rest[tail].mean()),
                to_17=first_below(rest_curve, TARGET_DB + 3), to_20=first_below(rest_curve, TARGET_DB),
                levels=[time_to(change_curve, level) for level in LEVELS + [TARGET_DB]],
                mean_db=10*np.log10(num_change.mean()/den_change.mean()),
                rest_curve=rest_curve, curve=change_curve)


def show_excess(rows):
    print(f"{'':<34}{'at rest':-^37}  {'after the flip, excess error':-^40}")
    print(f"{'filter':<22}{'eps or v':>12}{'mis.':>8}{'excess':>8}{'gap':>7}{'-17 dB':>7}{'-20 dB':>7}"
          + "".join(f"{level:>8.0f}" for level in LEVELS + [TARGET_DB]) + f"{'mean mis.':>11}{'mean exc.':>11}")
    for name, r in rows.items():
        x = r["excess"]
        value = "grid" if np.isnan(r["value"]) else f"{r['value']:.4e}"
        print(f"{name:<22}{value:>12}{r['floor']:>8.2f}{x['floor']:>8.2f}{x['floor'] - r['floor']:>7.2f}"
              f"{x['to_17']:>7d}{x['to_20']:>7d}" + "".join(f"{s:>8d}" for s in x["levels"])
              + f"{r['mean_db']:>11.2f}{x['mean_db']:>11.2f}")
    print("   floors and means in dB; mis. = misalignment, excess = excess error, gap = excess - mis.")
    print("   steps on the excess error (smoothed, good to about 50 steps); -1 = never within the run")


def run_filter(name, value, b_eta, X, D, flip_at=None):
    # misalignment and s_t, (B, n) each
    family, method = name.split(", ")
    if method == "bank":
        out = bank_batch(X, D, ho, EPS_BANK if family == "sKF" else V_BANK, Q_BANK, family, chi_laplacian,
                         log_density_laplacian, b_eta, flip_at=flip_at, excess=True)
        return out["mis"], out["s"]
    values = np.full(len(X), value)
    if family == "sKF":
        return sKF_batch(X, D, ho, b_eta, values, chi_laplacian, flip_at=flip_at, excess=True)
    if family == "fKF":
        return fKF_batch(X, D, ho, b_eta, values, chi_laplacian, flip_at=flip_at, excess=True)
    mis, _, s = KF_batch(X, D, ho, b_eta, values, chi_laplacian, flip_at=flip_at, excess=True)
    return mis, s


# floor, own + 3, recovery and mean over the record printed by bank-skf-fkf.ipynb (commit 57bd568)
BANK_NB = {(-0.9, "sKF, exact"): (-20.01, 4756, 7584, -14.08), (-0.9, "sKF, bank"): (-25.92, 68043, 75315, -13.89),
           (-0.9, "fKF, exact"): (-19.99, 5461, 8188, -13.45), (-0.9, "fKF, bank"): (-15.68, 28234, 29311, -11.92),
           (0.0, "sKF, exact"): (-20.01, 1564, 1221, -17.22), (0.0, "sKF, bank"): (-44.92, 6963, 6433, -22.64),
           (0.0, "fKF, exact"): (-20.00, 1056, 1596, -16.67), (0.0, "fKF, bank"): (-44.58, 12135, 12233, -23.01)}


def scenario_run(ar_a, tuned):
    """The KF tuned to -20 dB, then the five filters at rest and across the change, with s_t."""
    start = time.time()
    (X_r, D_r), (X_c, D_c) = scenario_signals(ar_a)
    b_eta = noise_of(ar_a)[1]
    den_r = (clean_output(X_r)**2).mean(axis=0)
    den_c = (clean_output(X_c, change_at=N)**2).mean(axis=0)
    print(f"mean of (x_t' theta_t)^2 over the change runs: {den_c[M:].mean():.4f}; "
          f"h' R_xx h = {signal_power(ar_a):.4f}")
    floors, settled = sweep_kf(b_eta, X_r, D_r)
    eps_kf, status = pick(EPS_GRID, floors, settled, TARGET_DB)
    print("KF: grid floors " + " ".join(f"{f:.1f}" + ("" if s else "*") for f, s in zip(floors, settled))
          + "   (* not settled)")
    print(f"   at -20 dB: eps = {eps_kf:.4e}, {status}   [{time.time() - start:.0f} s]")
    values = {"sKF, exact": tuned["sKF"], "sKF, bank": np.nan, "fKF, exact": tuned["fKF"], "fKF, bank": np.nan,
              "KF, exact": eps_kf}
    rows = {}
    for name, value in values.items():
        mis_r, s_r = run_filter(name, value, b_eta, X_r, D_r)
        mis_c, s_c = run_filter(name, value, b_eta, X_c, D_c, flip_at=N)
        rows[name] = dict(value=value, rest_curve=mis_r.mean(axis=0), **measure(mis_r.mean(axis=0), mis_c.mean(axis=0)),
                          excess=measure_excess(s_r, s_c, den_r, den_c))
        print(f"{name} done [{time.time() - start:.0f} s]")
    print("\ncontrol: the four filters of the bank notebook against its printed numbers (ours / printed)")
    for name, r in rows.items():
        if (ar_a, name) in BANK_NB:
            floor, steps, rec, mean = BANK_NB[(ar_a, name)]
            print(f"   {name:<11} floor {r['floor']:.2f} / {floor:.2f}, own+3 {r['reached']} / {steps}, "
                  f"recovery {r['rec']} / {rec}, mean {r['mean_db']:.2f} / {mean:.2f}")
    print("\nmisalignment")
    show_table(rows)
    print("\nexcess error")
    show_excess(rows)
    return rows


print("measurement ready")

## 6. AR($-0.9$)

Exact sKF and fKF at the values tuned to $-20$ dB in the bank notebook ($\varepsilon = 1.3493 \times 10^{-6}$,
$v = 2.985 \times 10^{-4}$), the banks as there ($q = 10^{-4}$, $v_0 = 2$), and the full KF at its own $-20$ dB value.

In [ ]:
rows_ar = scenario_run(-0.9, {"sKF": 1.3493e-6, "fKF": 2.985e-4})

## 7. White input

The same, with the white-input values of the bank notebook ($\varepsilon = 1.319 \times 10^{-6}$, $v = 3.017 \times 10^{-4}$).

In [ ]:
rows_w = scenario_run(0.0, {"sKF": 1.319e-6, "fKF": 3.017e-4})

## 8. Figures

**Figure 1.** Across the change, time counted from it. Rows: AR($-0.9$), white input. Left: misalignment
$\lVert\boldsymbol{\theta}_t - \boldsymbol{w}_{t-1}\rVert^2$; right: excess error
$\mathrm{E}[s_t^2]/\mathrm{E}[(\boldsymbol{x}_t^{\mathsf T}\boldsymbol{\theta}_t)^2]$ (101-step moving average). Colour = filter
(exact: orange, bank: violet, full KF: blue; there is no minorized filter here, which is blue in the other notebooks),
line style = family (sKF solid, fKF dashed). Grey: $-20$ dB. Black, dash-dot: the change. The same $y$ range on both
panels of a row.

**Figure 2.** The same from zero, at rest (the first $N$ steps of the rest run).

In [ ]:
LOOK = {"sKF, exact": ("#eb6834", "-"), "fKF, exact": ("#eb6834", "--"), "sKF, bank": ("#4a3aa7", "-"),
        "fKF, bank": ("#4a3aa7", "--"), "KF, exact": ("#2a78d6", "-")}
MAX_POINTS = 25000          # longer curves are thinned for the svg


def plot_runs(which, title, panels):
    """which = "change": across the flip, time from it; "rest": from zero."""
    fig, axes = plt.subplots(2, 2, figsize=(12, 8.5), constrained_layout=True, sharey="row")
    for row, (label, rows, t_max, ylim) in enumerate(panels):
        t, lo = ((np.arange(2*N) - N)/FS, -0.25) if which == "change" else (np.arange(N)/FS, 0.0)
        keep = np.flatnonzero((t >= lo) & (t <= t_max))
        keep = keep[::max(1, len(keep)//MAX_POINTS)]
        for col, metric in enumerate(("misalignment", "excess error")):
            ax = axes[row, col]
            for name, r in rows.items():
                source = r if col == 0 else r["excess"]
                curve = source["curve"] if which == "change" else source["rest_curve"]
                colour, style = LOOK[name]
                ax.plot(t[keep], 10*np.log10(curve[keep]), color=colour, ls=style, lw=1.4, label=name)
            if which == "change":
                ax.axvline(0, color="k", ls="-.", lw=0.9)
            ax.axhline(TARGET_DB, color="0.45", lw=0.8)
            ax.set(xlim=(lo, t_max), ylim=ylim, title=f"{label}, {metric}",
                   xlabel="time from the change [s]" if which == "change" else "time [s]")
            ax.grid(alpha=0.25)
        axes[row, 0].set_ylabel(r"$\|\theta_t - w_{t-1}\|^2$ [dB]")
        axes[row, 1].set_ylabel(r"$\mathrm{E}[s_t^2]\,/\,\mathrm{E}[(x_t^T\theta_t)^2]$ [dB]")
    axes[0, 0].legend(fontsize=8, loc="upper right")
    fig.suptitle(title)
    plt.show()


plot_runs("change", r"Across $h_o \to -h_o$, $\beta^* = 0.2$, 5 dB, $b_\eta = \mathrm{E}|\eta|$",
          [("AR(-0.9)", rows_ar, 2.5, (-42, 8)), ("white input", rows_w, 0.6, (-50, 8))])
plot_runs("rest", r"From zero, $\beta^* = 0.2$, 5 dB, $b_\eta = \mathrm{E}|\eta|$",
          [("AR(-0.9)", rows_ar, N/FS, (-42, 8)), ("white input", rows_w, 2.0, (-50, 8))])

## 9. The short spikes in the two metrics

Two numbers in the tables come from the fKF bank's short spikes (bank notebook, section 8.1: two outliers move the
weight to $v = 10^{-2}$ for a few steps):
- with white input its excess floor is 1.6 dB below its misalignment floor, where the introduction predicts 0 dB;
- with AR($-0.9$) its spikes are much taller in the excess error than in the misalignment (Figures 1 and 2).

In a spike the output is the filter that has just made a large correction along $\boldsymbol{x}_{t-1}$, so its error
$\boldsymbol{\Delta}$ lies along $\boldsymbol{x}_{t-1}$, and
$s_t \approx \lVert\boldsymbol{\Delta}\rVert\,\boldsymbol{x}_t^{\mathsf T}\boldsymbol{x}_{t-1}/\lVert\boldsymbol{x}_{t-1}\rVert$.
For those steps $\boldsymbol{w}_{t-1}$ is not independent of $\boldsymbol{x}_t$, as the introduction assumed.
- AR($-0.9$): consecutive windows are nearly parallel, $\boldsymbol{x}_t^{\mathsf T}\boldsymbol{x}_{t-1} \approx \rho M$ with
  the lag-1 correlation $\rho \approx -0.9$, so $s_t^2 \approx \rho^2 M\lVert\boldsymbol{\Delta}\rVert^2$. Over the spike steps
  the excess error should be about $\rho^2 M/\boldsymbol{h}_o^{\mathsf T}\boldsymbol{R}_{xx}\boldsymbol{h}_o \approx 130$ times the
  misalignment.
- White input: $\boldsymbol{x}_t^{\mathsf T}\boldsymbol{x}_{t-1}$ is about $\pm\sqrt{M}$, so on average $s_t^2 = \lVert\boldsymbol{\Delta}\rVert^2$,
  the same weight in both metrics. But each spike step gives one random draw of $s_t^2$ (roughly $\chi^2_1$), while its
  misalignment is exact. With few spike steps the two can differ a lot.

The test: the rest runs of both banks again. A spike step is one where the misalignment of a realisation is above
10 times its own median over the last quarter. Printed: how many there are and in how many realisations; their
effective number $(\sum m)^2/\sum m^2$ ($m$ = their misalignments); their excess error over their misalignment; their
share of each metric; and the floors with and without them.

In [ ]:
start = time.time()
for ar_a in (0.0, -0.9):
    (X_r, D_r), _ = scenario_signals(ar_a)
    den_tail = (clean_output(X_r)**2).mean(axis=0)[3*N//4:].mean()
    lag1 = np.mean(X_r[:, 1:]*X_r[:, :-1])/np.mean(X_r**2)          # rho, the input's lag-1 correlation
    print(f"AR {ar_a:>4}: rho = {lag1:+.3f}, rho^2 M / h' R_xx h = {lag1**2*M/signal_power(ar_a):.1f}")
    for family, grid in (("sKF", EPS_BANK), ("fKF", V_BANK)):
        out = bank_batch(X_r, D_r, ho, grid, Q_BANK, family, chi_laplacian, log_density_laplacian, noise_of(ar_a)[1],
                         excess=True)
        mis, s2 = out["mis"][:, 3*N//4:], out["s"][:, 3*N//4:]**2
        spike = mis > 10*np.median(mis, axis=1, keepdims=True)
        m = mis[spike]
        print(f"   {family} bank: {spike.sum()} spike steps of {spike.size}, in {np.any(spike, axis=1).sum()} realisations")
        if spike.any():
            print(f"      effective number {m.sum()**2/(m**2).sum():.1f}; over them the excess error is "
                  f"{s2[spike].sum()/den_tail/m.sum():.2f} times the misalignment; their share: "
                  f"{m.sum()/mis.sum():.2f} of the misalignment, {s2[spike].sum()/s2.sum():.2f} of E[s^2]")
        floors = [10*np.log10(mis[keep].mean()) for keep in (np.ones_like(spike), ~spike)]
        excess = [10*np.log10(s2[keep].mean()/den_tail) for keep in (np.ones_like(spike), ~spike)]
        print(f"      all steps:          misalignment {floors[0]:.2f}, excess {excess[0]:.2f}, gap {excess[0] - floors[0]:+.2f} dB")
        print(f"      without the spikes: misalignment {floors[1]:.2f}, excess {excess[1]:.2f}, gap {excess[1] - floors[1]:+.2f} dB")
del X_r, D_r, out
print(f"[{time.time() - start:.0f} s]")

## 10. Findings

**Checks: all pass.**
1. With the Gaussian $\chi$, `KF_batch` (with the two options added) is the textbook Kalman filter across a flip of
   $\boldsymbol{h}_o$: misalignment to $5 \times 10^{-14}$, $s_t$ to $3 \times 10^{-14}$ (relative).
2. $d_t - \boldsymbol{x}_t^{\mathsf T}\boldsymbol{\theta}_t$ is the noise drawn again, to $4 \times 10^{-15}$. Over the 20 change runs
   the mean of $(\boldsymbol{x}_t^{\mathsf T}\boldsymbol{\theta}_t)^2$ is 0.8177 against
   $\boldsymbol{h}_o^{\mathsf T}\boldsymbol{R}_{xx}\boldsymbol{h}_o = 0.8173$ (AR) and 1.0003 against 1 (white).
3. The banks' $s_t$ equals the one formed from their recorded outputs to $10^{-15}$. A bank with $K = 1$ gives the
   single filter's $s_t$ and misalignment bit for bit.
4. Control: the four filters of the bank notebook give its printed floor, steps, recovery and mean exactly, in both
   scenarios.

**1. The full KF (Leszek's first item): whitening does most of it.**

The KF is tuned to $-20$ dB like the others: $\varepsilon = 8.86 \times 10^{-7}$ (AR) and $1.66 \times 10^{-6}$ (white).
Steps are notebook 09's: to the own floor + 3 dB from zero, and back to it after the flip. All three filters have a
$-20$ dB floor.

| | from zero, AR | from zero, white | AR / white | after the flip, AR | after the flip, white | AR / white | mean AR [dB] | mean white [dB] |
|---|---|---|---|---|---|---|---|---|
| exact sKF | 4 756 | 1 564 | 3.04 | 7 584 | 1 221 | 6.21 | $-14.08$ | $-17.22$ |
| exact fKF | 5 461 | 1 056 | 5.17 | 8 188 | 1 596 | 5.13 | $-13.45$ | $-16.67$ |
| **full KF** | **1 254** | **603** | **2.08** | **1 972** | **791** | **2.49** | $\mathbf{-15.63}$ | $\mathbf{-17.65}$ |

- With AR($-0.9$) the KF is 3.8 times faster than the exact sKF, from zero (1 254 against 4 756 steps) and after the
  flip (1 972 against 7 584). Its mean over the record is 1.5 dB better ($-15.63$ against $-14.08$), and better than
  both banks ($-13.89$, $-11.92$).
- The KF's colour penalty (AR over white) is 2.1 from zero and 2.5 after the flip, against 3.0 / 6.2 for the sKF and
  5.2 / 5.1 for the fKF. Whitening takes away most of the penalty, not all of it.
- From zero, the KF with AR input is faster than the sKF with white input (1 254 against 1 564 steps), as
  `vkf-kf.ipynb` found at the old $b_\eta$. After the flip it is slower (1 972 against 1 221).
- With white input the KF is still the fastest single filter (603 against 1 564 / 1 056 from zero, 791 against
  1 221 / 1 596 after the flip), but its mean is only 0.4 dB better than the sKF's, and the banks stay ahead on the
  mean ($-22.64$ / $-23.01$).

**2. The excess error (Leszek's second item): with AR input the banks' remaining error is where the input has little
power.**

How to read the gap (excess floor minus misalignment floor): an error spread evenly over all directions gives
$\mathrm{E}[s_t^2]$ = the misalignment (unit input variance), while the denominator is
$\boldsymbol{h}_o^{\mathsf T}\boldsymbol{R}_{xx}\boldsymbol{h}_o = 0.817$ instead of $\lVert\boldsymbol{h}_o\rVert^2 = 1$. So an even
spread gives a gap of $10\log_{10}(1/0.817) = +0.9$ dB with AR input, and a gap below that means the error sits where
the input has little power.

AR($-0.9$), floors and means in dB, steps on the excess error:

| | misalignment floor | excess floor | gap | from zero to $-17$ dB | after the flip to $-10$ / $-15$ dB | mean misalignment | mean excess |
|---|---|---|---|---|---|---|---|
| exact sKF | $-20.01$ | $-19.00$ | $+1.0$ | 2 905 | 2 249 / 4 255 | $-14.08$ | $-17.01$ |
| sKF bank | $-25.92$ | $-36.24$ | $-10.3$ | 2 163 | 1 597 / 2 396 | $-13.89$ | $-20.72$ |
| exact fKF | $-19.99$ | $-18.91$ | $+1.1$ | 3 288 | 2 503 / 4 605 | $-13.45$ | $-16.90$ |
| fKF bank | $-15.68$ | $-23.99$ | $-8.3$ | 2 153 | 1 951 / 2 515 | $-11.92$ | $-17.63$ |
| full KF | $-20.01$ | $-25.16$ | $-5.2$ | 685 | 884 / 1 151 | $-15.63$ | $-20.79$ |

- The exact sKF and fKF have gaps of $+1.0$ and $+1.1$ dB, close to the $+0.9$ dB of an even spread: at their floor
  their error is spread over all directions.
- The banks have gaps of $-10.3$ and $-8.3$ dB ($-10.7$ without the fKF bank's spikes, section 9). Most of what is
  left of their misalignment lies where the AR($-0.9$) input has little power, so their prediction errors hardly see
  it. This is the draft's explanation of why the banks crawl with coloured input (bank notebook, section 9), now
  measured. It fits the posterior going back to the smallest grid value while the misalignment is still high (bank
  notebook, Figure 2); this notebook does not test that link.
- Measured by the excess error, the banks beat the single filters at every level after the flip ($-10$ dB at 1 597 /
  1 951 steps against 2 249 / 2 503; $-15$ dB at 2 396 / 2 515 against 4 255 / 4 605). The sKF bank's mean excess
  error is 3.7 dB better than the exact sKF's ($-20.72$ against $-17.01$) and level with the full KF's ($-20.79$). By
  the misalignment the same bank is no better than the exact sKF ($-13.89$ against $-14.08$).
- The full KF also leaves its error where the input has little power (gap $-5.2$ dB), at the same $-20$ dB
  misalignment as the single filters. It is the fastest filter in both metrics.
- The steps to $-20$ dB of excess error of the single filters (8 908 and 17 353 from zero, 15 334 and 15 335 after the
  flip) mean little: their excess floor is $-19$ dB, and those are dips of the smoothed curve below it.

**White input: the excess error is the misalignment.** Floors agree to 0.15 dB and means to 0.1 dB, for every
filter except the fKF bank's floor, which is 1.6 dB lower because of its spikes (section 9). With white input the
excess error adds nothing: the banks' gain there shows in both metrics.

**The spikes (section 9).**
- White input: 41 spike steps of the fKF bank, in 8 realisations, carry 72 % of its misalignment over the last
  quarter. Without them the gap is 0.00 dB, and the floor is $-50.0$ dB, not $-44.6$. Over those steps $s_t^2$ came out
  at 0.57 times the misalignment, where 1 is expected on average. With an effective number of 15 draws, that is within
  the scatter (relative standard deviation about $\sqrt{2/15} \approx 0.4$).
- AR($-0.9$): 8 spike steps carry none of the misalignment but 43 % of $\mathrm{E}[s_t^2]$. Over them the excess error
  is 157 times the misalignment, the same order as the 127 that $\rho^2 M/\boldsymbol{h}_o^{\mathsf T}\boldsymbol{R}_{xx}\boldsymbol{h}_o$
  gives. Without them the excess floor is $-26.42$ dB.
- So a few outliers set the fKF bank's floor in both scenarios: in the misalignment with white input, in the excess
  error with AR input. The sKF bank has no spike steps.

**Open.**
- Which metric to report for the banks with coloured input. By the misalignment they do no better than the tuned
  single filters. By the excess error they do clearly better, and the sKF bank matches the full KF. Both are measured
  here.

**Not done.**
- A bank of full KFs (not in the draft; it needs a merge rule for $\boldsymbol{\Sigma}$).
- The minorized and matched filters (choice made before writing code).
- Tuning on the excess error: every filter is tuned on the misalignment, as in the bank notebook.

In [ ]:
print(f"whole notebook: {(time.time() - NOTEBOOK_START)/60:.1f} min")